In [2]:
import os
import cv2
import numpy as np

def ensure_dir(directory):
    """Создает папку, если ее не существует."""
    if not os.path.exists(directory):
        os.makedirs(directory)

def mosaic_approximation(window, tile_factor=4, threshold=128):
    """
    Аппроксимирует форму заданного окна с помощью мозаичного представления.
    
    Параметры:
      window     - окно изображения (градации серого)
      tile_factor- количество блоков по каждой оси (окно делится на tile_factor x tile_factor блоков)
      threshold  - порог для бинаризации каждого блока
      
    Для каждого блока вычисляется среднее значение; если оно превышает threshold, то блок
    заполняется 255, иначе 0.
    
    Возвращает бинаризованное мозаичное изображение того же размера, что и window.
    """
    h, w = window.shape[:2]
    block_h = h // tile_factor
    block_w = w // tile_factor
    mosaic = np.zeros_like(window)
    
    for i in range(tile_factor):
        for j in range(tile_factor):
            start_y = i * block_h
            start_x = j * block_w
            end_y = (i+1) * block_h if i < tile_factor - 1 else h
            end_x = (j+1) * block_w if j < tile_factor - 1 else w
            block = window[start_y:end_y, start_x:end_x]
            mean_val = np.mean(block)
            binary_val = 255 if mean_val > threshold else 0
            mosaic[start_y:end_y, start_x:end_x] = binary_val
    return mosaic

def sliding_window_morph_diff(f_img, g_img, window_size=32, step=16, tile_factor=4,
                              threshold_diff=0.3, bin_thresh=128):
    """
    Реализует вариант 2 выделения отличий по форме.
    
    Для каждого положения скользящего окна в изображении f:
      - Извлекается окно f_win и соответствующий фрагмент g_win из g.
      - Для окна f_win строится мозаичная аппроксимация (функция mosaic_approximation).
      - Фрагмент g_win бинаризуется с порогом bin_thresh.
      - Вычисляется абсолютная разность между мозаичным представлением f_win и бинаризованным g_win.
      - Если доля отличающихся пикселей (норма разности) превышает threshold_diff,
        центр окна помечается как область отличий.
    
    Параметры:
      f_img, g_img   - исходные изображения (градации серого, одинаковый размер)
      window_size    - размер окна
      step           - шаг скользящего окна (можно задавать меньше, чем window_size, для перекрытия)
      tile_factor    - деление окна на (tile_factor x tile_factor) блоков для мозаичного аппроксимирования
      threshold_diff - порог для нормы разности (от 0 до 1)
      bin_thresh     - порог бинаризации для фрагмента g
      
    Возвращает:
      - result: изображение f с нарисованными красными кружками в центрах окон, где обнаружены отличия
      - diff_map: карта различий (бинаризованное изображение, где окна с отличиями помечены белым)
    """
    h_img, w_img = f_img.shape[:2]
    result = np.copy(f_img)
    if len(result.shape) == 2:
        # Для возможности рисовать цветными отметками переводим в BGR
        result = cv2.cvtColor(result, cv2.COLOR_GRAY2BGR)
    
    diff_map = np.zeros_like(f_img, dtype=np.uint8)
    
    for y in range(0, h_img - window_size + 1, step):
        for x in range(0, w_img - window_size + 1, step):
            f_win = f_img[y:y+window_size, x:x+window_size]
            g_win = g_img[y:y+window_size, x:x+window_size]
            
            # Аппроксимация формы окна f с помощью мозаики
            mosaic_f = mosaic_approximation(f_win, tile_factor=tile_factor, threshold=bin_thresh)
            
            # Бинаризация окна g
            _, g_win_bin = cv2.threshold(g_win, bin_thresh, 255, cv2.THRESH_BINARY)
            
            # Вычисление абсолютной разности (морфологическая разность)
            diff = cv2.absdiff(mosaic_f, g_win_bin)
            # Нормируем: доля пикселей, где имеются отличия
            diff_norm = np.sum(diff > 0) / (window_size * window_size)
            
            # Если норма отличается значимо, отмечаем окно
            if diff_norm > threshold_diff:
                center_x = x + window_size // 2
                center_y = y + window_size // 2
                cv2.circle(result, (center_x, center_y), radius=3, color=(0, 0, 255), thickness=-1)
                diff_map[y:y+window_size, x:x+window_size] = 255
    return result, diff_map

def main():
    # Пути к изображениям f и g
    # Изображения можно разместить в папке "source_images"
    # Здесь предполагается, что файлы называются "f.png" и "g.png"
    f_path = os.path.join("source_images", "f.png")
    g_path = os.path.join("source_images", "g.png")
    
    if not os.path.exists(f_path) or not os.path.exists(g_path):
        print("Изображения f.png и/или g.png не найдены в папке source_images")
        return
    
    f_img = cv2.imread(f_path, cv2.IMREAD_GRAYSCALE)
    g_img = cv2.imread(g_path, cv2.IMREAD_GRAYSCALE)
    
    # Проверка, что изображения одинакового размера
    if f_img.shape != g_img.shape:
        print("Изображения f и g должны быть одинакового размера")
        return
    
    # Параметры: можно изменить в зависимости от характеристик изображений
    window_size = 32      # размер скользящего окна
    step = 16             # шаг скольжения окна
    tile_factor = 4       # деление окна на 4x4 блока для мозаики
    threshold_diff = 0.3  # если более 30% пикселей отличаются – отмечаем окно
    bin_thresh = 128      # порог бинаризации
    
    result_img, diff_map = sliding_window_morph_diff(f_img, g_img, window_size,
                                                     step, tile_factor, threshold_diff, bin_thresh)
    
    # Создаем папку для результатов
    output_folder = os.path.join("results", "task7_variant2")
    ensure_dir(output_folder)
    
    result_path = os.path.join(output_folder, "result.png")
    diff_map_path = os.path.join(output_folder, "diff_map.png")
    
    cv2.imwrite(result_path, result_img)
    cv2.imwrite(diff_map_path, diff_map)
    
    print("Обработка завершена. Результаты сохранены в", output_folder)

if __name__ == "__main__":
    main()


Обработка завершена. Результаты сохранены в results/task7_variant2
